# ==============================================================================
# 💎 GEMOLOGICAL VALUATION: DIAMOND PRICE VIA RANDOM FOREST REGRESSOR
# ==============================================================================
### Mathematical Formulation & Theoretical Context:
Diamond pricing depends non-linearly on the "4 Cs": Carat, Cut, Color, and Clarity.
Random Forest Regressor aggregates $B$ independent regression trees trained on bootstrap samples:
$$\hat{y}(\mathbf{x}) = \frac{1}{B} \sum_{b=1}^B \hat{f}_b(\mathbf{x})$$

At each internal node, candidate splits are evaluated on a random subspace of $m = p/3$ features,
maximizing variance reduction while decorrelating tree structures.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Gemological Valuation Environment ready.")


✅ STEP 1: Gemological Valuation Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle Diamonds dataset (53,940 diamonds).


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/diamonds/diamonds.csv'
df = pd.read_csv(data_path)

# Drop index column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Sample 15,000 diamonds for efficient, rigorous cross-validation while maintaining variance
df_sample = df.sample(n=15000, random_state=42).reset_index(drop=True)

df_sample.columns = [c.strip().lower() for c in df_sample.columns]
print(f"📊 Dataset Shape: {df_sample.shape[0]:,} diamonds, {df_sample.shape[1]} attributes")
print(f"Price Summary:\n{df_sample['price'].describe().round(2)}")
df_sample.head(3)


📊 Dataset Shape: 15,000 diamonds, 10 attributes
Price Summary:
count    15000.00
mean      3890.70
std       3947.43
min        326.00
25%        938.00
50%       2400.00
75%       5251.00
max      18823.00
Name: price, dtype: float64


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.24,Ideal,G,VVS1,62.1,56.0,559,3.97,4.00,2.47
1,0.58,Very Good,F,VVS2,60.0,57.0,2201,5.44,5.42,3.26
2,0.40,Ideal,E,VVS2,62.1,55.0,1238,4.76,4.74,2.95


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `price`. Filtering invalid zero dimensions ($x, y, z > 0$).


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
df_sample = df_sample[(df_sample['x'] > 0) & (df_sample['y'] > 0) & (df_sample['z'] > 0)].copy()

X = df_sample.drop(columns=['price'])
y = df_sample['price']

print(f"Cleaned instances: {X.shape[0]:,}")
print(f"Missing attributes: {X.isnull().sum().sum()}")


Cleaned instances: 14,999
Missing attributes: 0


## 🔒 STEP 6: LEAK-FREE TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Diamonds: {X_train.shape[0]:,}")
print(f"Test Diamonds : {X_test.shape[0]:,}")


Train Diamonds: 11,999
Test Diamonds : 3,000


## ⚙️ STEP 7: ORDINAL CATEGORICAL PREPROCESSOR
Cut, color, and clarity possess natural ordered hierarchies.
We use `OrdinalEncoder` to preserve quality rankings without one-hot dimensionality explosion.


In [5]:
# STEP 7: PREPROCESSOR
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

ordinal_cols = ['cut', 'color', 'clarity']
num_cols = ['carat', 'depth', 'table', 'x', 'y', 'z']

preprocessor = ColumnTransformer([
    ('ord', OrdinalEncoder(categories=[cut_order, color_order, clarity_order]), ordinal_cols),
    ('num', 'passthrough', num_cols)
])
print("✅ STEP 7: Gemological Ordinal Preprocessor configured.")


✅ STEP 7: Gemological Ordinal Preprocessor configured.


## ⚠️ STEP 8: SINGLE TREE VS ENSEMBLE FOREST DUEL
Demonstrating how a forest of 150 trees cuts variance compared to a single decision tree.


In [6]:
# STEP 8: BASELINE DUEL
pipe_single = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeRegressor(random_state=42))
])
pipe_single.fit(X_train, y_train)

pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])
pipe_rf.fit(X_train, y_train)

r2_single = r2_score(y_test, pipe_single.predict(X_test))
r2_rf = r2_score(y_test, pipe_rf.predict(X_test))

print(f"Single Decision Tree R² : {r2_single*100:.2f}% | RMSE: ${np.sqrt(mean_squared_error(y_test, pipe_single.predict(X_test))):.2f}")
print(f"Random Forest Ensemble R²: {r2_rf*100:.2f}% | RMSE: ${np.sqrt(mean_squared_error(y_test, pipe_rf.predict(X_test))):.2f}")
print(f"Ensemble Improvement   : +{(r2_rf - r2_single)*100:.2f}% R² gain!")


Single Decision Tree R² : 95.49% | RMSE: $830.69
Random Forest Ensemble R²: 97.90% | RMSE: $567.25
Ensemble Improvement   : +2.41% R² gain!


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 3-FOLD CV
Optimizing `max_depth` and `min_samples_leaf` for valuation $R^2$.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'rf__n_estimators': [100, 150],
    'rf__max_depth': [12, 16],
    'rf__min_samples_leaf': [2, 5]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1))
])

grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 3-Fold CV R²   : {grid.best_score_*100:.2f}%")


🏆 Best Hyperparameters: {'rf__max_depth': 16, 'rf__min_samples_leaf': 2, 'rf__n_estimators': 150}
🎯 Best 3-Fold CV R²   : 97.68%


## ⚔️ STEP 10: ALGORITHM DUEL (Random Forest vs Single Tree)
Quantifying precision improvements.


In [8]:
# STEP 10: MODEL BENCHMARK
y_pred_tree = pipe_single.predict(X_test)
y_pred_rf = best_model.predict(X_test)

print("="*65)
print(f"Single Decision Tree -> R²: {r2_score(y_test, y_pred_tree)*100:.2f}% | MAE: ${mean_absolute_error(y_test, y_pred_tree):.2f}")
print(f"Random Forest (Tuned)-> R²: {r2_score(y_test, y_pred_rf)*100:.2f}% | MAE: ${mean_absolute_error(y_test, y_pred_rf):.2f}")
print("="*65)


Single Decision Tree -> R²: 95.49% | MAE: $402.62
Random Forest (Tuned)-> R²: 97.96% | MAE: $288.72


## 📊 STEP 11: RESIDUAL ANALYSIS & VALUATION METRICS
Reviewing gem appraisal errors and pricing drivers.


In [9]:
# STEP 11: EVALUATION REPORT
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"📋 GEMOLOGICAL VALUATION METRICS:")
print(f"   Mean Absolute Error (MAE) : ${mae:,.2f}")
print(f"   Root Mean Squared Error   : ${rmse:,.2f}")
print(f"   R² Coefficient            : {r2_score(y_test, y_pred_rf)*100:.2f}%")


📋 GEMOLOGICAL VALUATION METRICS:
   Mean Absolute Error (MAE) : $288.72
   Root Mean Squared Error   : $559.21
   R² Coefficient            : 97.96%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & APPRAISAL SMOKE TEST
Deploying pipeline and validating auction floor instant appraisal latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/diamonds_random_forest_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_diamond = pd.DataFrame([{
    'carat': 1.15,
    'cut': 'Ideal',
    'color': 'E',
    'clarity': 'VS1',
    'depth': 61.8,
    'table': 56.0,
    'x': 6.75,
    'y': 6.79,
    'z': 4.18
}])

t0 = time.perf_counter()
pred_price = loaded_pipe.predict(sample_diamond)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n💎 LIVE AUCTION FLOOR GEM APPRAISAL:")
print(f"   Estimated Retail Value: 💵 ${pred_price:,.2f}")
print(f"   Appraisal Latency     : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/diamonds_random_forest_pipeline.joblib (15,839,282 bytes)



💎 LIVE AUCTION FLOOR GEM APPRAISAL:
   Estimated Retail Value: 💵 $9,345.95
   Appraisal Latency     : 44.365 ms (SLA < 2.0ms: CHECK)
